# 01 — EDA: BeeBuzz

Exploratory data analysis of the Mendeley Buzz Pollination dataset (200 events, 5 bee species).
Run `python src/eda.py` to regenerate figures in `reports/`.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
from features import load, FEATURES, TARGET_SPECIES, TARGET_TYPE

df = load('../data/raw/buzz_data.csv')
print(df.shape, '| nulls:', int(df.isnull().sum().sum()))
df[[TARGET_SPECIES, TARGET_TYPE, 'ffreq', 'duration.s']].head()

## 1. Class distribution (balanced by design)

5 species × 40 events each; 100 floral + 100 flight (20/20 per species).

In [ ]:
print(df[TARGET_SPECIES].value_counts())
print()
print(pd.crosstab(df[TARGET_SPECIES], df[TARGET_TYPE]))

## 2. Fundamental frequency by species and buzz type

`ffreq` is the top SHAP feature. Boxplots show overlap between species.

In [ ]:
from IPython.display import Image
Image('../reports/eda_ffreq.png')

## 3. Duration by species and buzz type

In [ ]:
Image('../reports/eda_duration.png')

## 4. MFCC profiles per species

Mean MFCC vectors differ by species — the signal the classifier uses.

In [ ]:
Image('../reports/eda_mfcc_heatmap.png')

## 5. Correlation matrix — redundant features

13 pairs with |corr| > 0.9, all `mean.mfc i` vs `med.mfc i` of the same coefficient (~0.999). Median duplicates the mean. This explains why LDA drops when adding features (multicollinearity) while RF is robust.

In [ ]:
Image('../reports/eda_corr.png')

## 6. PCA — species are not linearly separable in 2D

PC1 explains 18%, PC2 11%. No clean clusters: classification needs non-linear models (trees).

In [ ]:
Image('../reports/eda_pca.png')

## 7. Metadata / leakage checks

- `audio_files`: 200 unique (1 per row). Floral files start with `B_`, flight with `V_` — the filename prefix perfectly predicts buzz type. The 99% type accuracy must be read with this caveat: recording setup may differ between B_ and V_ sessions.
- `sample_numer`: 94 unique; all 100 flight rows share `44100` (likely a placeholder). Not a reliable individual ID → grouped CV by this column would collapse.
- `date`: includes Excel-serial-like strings ('44132'...). Data quality quirk, documented.
- `local`: single site (Reserva Ecológica do Panga). No site generalization claim.

In [ ]:
print('audio_files unique:', df['audio_files'].nunique())
print('sample_numer unique:', df['sample_numer'].nunique())
df['_prefix'] = df['audio_files'].str[0]
print(pd.crosstab(df['_prefix'], df[TARGET_TYPE]))